# Notebook 01 — Data Quality Assessment & Cleaning Pipeline
## NovaCred Credit Application Governance Audit

> **Executive Summary** — This notebook ingests 502 raw credit applications stored in a nested JSON file, systematically detects every data quality defect across 11 distinct issue types, applies the appropriate remediation (or a governance-safe flag where imputation would be inappropriate), and writes two output artefacts: a cleaned 500-row CSV ready for downstream analysis, and a machine-readable JSON quality report for the Governance Officer. Every cleaning step preserves the original raw value in a `_raw` column and records which rows were touched via a boolean `_flag` / `_missing` / `_imputed` column, creating a full audit trail without destroying evidence.

---

### Table of Contents

| § | Section | Issue | Data Quality Dimension |
|---|---------|-------|------------------------|
| 1 | [Imports & Configuration](#section-1) | — | — |
| 2 | [Load Raw Data & Flatten JSON](#section-2) | — | — |
| 3 | [Duplicate Record Detection](#section-3) | #1 Duplicate `_id` rows · #11 Conflicting SSNs | Uniqueness |
| 4 | [Inconsistent Gender Coding](#section-4) | #2 `M` / `F` abbreviations | Consistency |
| 5 | [Income Stored as Text](#section-5) | #3 String-typed numeric field | Validity / Data Type |
| 6 | [Missing Income Values](#section-6) | #4 `null` annual income | Completeness |
| 7 | [Negative Credit History Months](#section-7) | #5 Physically impossible negative values | Validity |
| 8 | [Impossible Debt-to-Income Ratio](#section-8) | #6 DTI > 1.0 (decimal point error) | Validity |
| 9 | [Mixed Date Formats](#section-9) | #7 Four competing formats in one column | Consistency |
| 10 | [Missing Date of Birth](#section-10) | #8 `null` DOB — PII, no imputation | Completeness |
| 11 | [Missing Email & SSN](#section-11) | #9 / #10 Missing PII fields | Completeness |
| 12 | [Data Quality Dashboard](#section-12) | All 11 issues summarised | — |
| 13 | [Export: CSV + JSON Report](#section-13) | — | — |
| 14 | [Pipeline Diagram](#section-14) | — | — |

---




### 1. Imports and configuration

In [342]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1: Import all the libraries (tools) we need
# ═══════════════════════════════════════════════════════════════════════════

# json: reads our raw data file which is stored in JSON format
import json

# re: "regular expressions" — a powerful way to check if text matches a pattern, We will use it to detect which date format a string is in (e.g. DD/MM/YYYY)
import re

# warnings: Python sometimes prints yellow warning messages that are not errors but clutter the output. This line lets us silence them.
import warnings

# numpy: fast numerical computing. We use it mainly for np.nan (a special "not a number" value) and np.polyfit (fitting a line through data points)
import numpy as np

# pandas: THE library for working with tables of data in Python. A DF is a basically a excel spreadsheet in memory.
import pandas as pd

# matplotlib & Seaborn: the foundational Python charting library and a popular extension that makes it easier to create nice-looking charts.
import matplotlib.pyplot as plt
import seaborn as sns

# datetime: Python's built-in module for working with dates.
from datetime import datetime

# Counter: a convenient tool that counts how many times each value appears
# in a list. E.g. Counter(['Male','Male','Female','M']) → {'Male':2,'Female':1,'M':1}
from collections import Counter


warnings.filterwarnings('ignore')

# ─── Set the visual style for all charts ────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)


# ─── File paths ─────────────────────────────────────────────────────────────
# Instead of typing the full file path over and over, we store them in variables once. If the file moves, we only change it here.

RAW_PATH    = '/Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/data/raw_credit_applications.json'  # the original, untouched file
CLEAN_PATH  = '/Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/data/cleaned_credit_applications.csv'  # where we save our clean output
REPORT_PATH = '/Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/data/data_quality_report.json'  # our written log of every issue found
FIGURES_DIR = '/Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/figures/'  # folder where charts are saved

# ─── Audit reference date ────────────────────────────────────────────────────
# We need a fixed "today" to calculate how old each applicant is from their
# date of birth. We use March 1, 2026 — the approximate date of this audit.
AUDIT_DATE = datetime(2026, 3, 1)

# A quick confirmation message so we know the cell ran without errors
print(f'✓ All libraries loaded successfully')
print(f'✓ Audit reference date set to: {AUDIT_DATE.strftime("%B %d, %Y")}')
print(f'✓ Raw data will be read from : {RAW_PATH}')
print(f'✓ Clean data will be saved to: {CLEAN_PATH}')


✓ All libraries loaded successfully
✓ Audit reference date set to: March 01, 2026
✓ Raw data will be read from : /Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/data/raw_credit_applications.json
✓ Clean data will be saved to: /Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/data/cleaned_credit_applications.csv


## Section 2  Loading the Raw Data & Converting It Into a Table


raw data currnetly is in `raw_credit_applications.json`. Here is what one credit application looks like inside the JSON file:

```json
{
  "_id": "app_001",
  "applicant_info": {
    "full_name": "Stephanie Nguyen",
    "email": "s.nguyen@gmail.com",
    "gender": "Female",
    "date_of_birth": "1990-03-15"
  },
  "financials": {
    "annual_income": 85000,
    "credit_history_months": 72,
    "debt_to_income": 0.24
  },
  "decision": {
    "loan_approved": true,
    "interest_rate": 4.2
  }
}

The data is **nested** . The application record contains `applicant_info`, which itself contains `full_name`, `email`, etc. Pandas cannot work directly with this structure, it needs a flat table where every piece of information is its own column.

In [343]:
# open(RAW_PATH, 'r') — opens the file at the path we defined above'r' means "read mode" — we are only reading, not writing to it

# The 'with' keyword ensures the file is automatically closed when we are done reading it, even if an error occurs. This prevents memory leaks.

# json.load(f) — reads the text inside the file and converts it into a Python object (in this case, a list of dictionaries — one per application)

with open(RAW_PATH, 'r') as f:
    raw_data = json.load(f)

print(f'Raw records loaded: {len(raw_data):,}')
print(f'Example record keys: {list(raw_data[0].keys())}')
print(f'applicant_info keys: {list(raw_data[0]["applicant_info"].keys())}')
print(f'financials keys    : {list(raw_data[0]["financials"].keys())}')
print(f'decision keys      : {list(raw_data[0]["decision"].keys())}')
print(f'spending_behavior  : {raw_data[0]["spending_behavior"][:2]}')
# spending_behavior is a LIST of dictionaries, not a single value  That is why it needs special handling when we flatten

Raw records loaded: 502
Example record keys: ['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp']
applicant_info keys: ['full_name', 'email', 'ssn', 'ip_address', 'gender', 'date_of_birth', 'zip_code']
financials keys    : ['annual_income', 'credit_history_months', 'debt_to_income', 'savings_balance']
decision keys      : ['loan_approved', 'rejection_reason']
spending_behavior  : [{'category': 'Shopping', 'amount': 480}, {'category': 'Rent', 'amount': 790}]


In [344]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3: Define the function that flattens one JSON record into one table row
# ═══════════════════════════════════════════════════════════════════════════

def flatten_record(record: dict) -> dict:
    """
    Takes one nested JSON credit application and returns a flat dictionary.

    Parameters
    ----------
    record : dict
        One raw application record as loaded from JSON.

    Returns
    -------
    dict
        A flat dictionary where every value is a simple scalar (number,
        string, boolean, or None) — ready to become one row in a DataFrame.
    """

    # ── Step A: Extract each nested sub-section into its own variable ────────
    #
    # record.get('applicant_info', {}) means:
    #   "give me the value stored under the key 'applicant_info'"
    #   "if that key doesn't exist, return an empty dictionary {} instead of crashing"

    ai  = record.get('applicant_info', {})   # personal details
    fin = record.get('financials', {})        # financial details
    dec = record.get('decision', {})          # loan decision
    sb  = record.get('spending_behavior', []) # list of spending entries (default: empty list)

    # ── Step B: Summarise the spending_behavior list ─────────────────────────
    #
    # spending_behavior is a LIST like this:
    # [{"category": "Rent", "amount": 800}, {"category": "Alcohol", "amount": 120}]
    #
    # We cannot put a list inside a single table cell, so we compute two scalar
    # summaries from it:
    #   1. total_spend  — sum of ALL spending categories
    #   2. alcohol_spend — sum of ONLY the "Alcohol" category

    # sum() adds up all the values we give it.
    # The expression inside is a "generator" — it loops over every item in sb,
    # and for each item checks:
    #   - Does item have an 'amount'? → item.get('amount', 0) returns 0 if missing
    #   - Is that amount actually a number? → isinstance(..., (int, float)) returns True/False.

    total_spend = sum(
        item.get('amount', 0)
        for item in sb
        if isinstance(item.get('amount'), (int, float))
    )

    # alcohol_spend: sum of ONLY entries where category == 'Alcohol'
    # Same pattern as total_spend, but with an extra filter:
    #   item.get('category') == 'Alcohol'  →  only include this item if it is the Alcohol category.
    # This is a useful risk signal: high alcohol spend relative to income
    # can be a flag in credit-risk modelling.
    alcohol_spend = sum(
        item.get('amount', 0)
        for item in sb
        if isinstance(item.get('amount'), (int, float))
        and item.get('category') == 'Alcohol'
    )

    # ── Step C: Build and return the flat dictionary ─────────────────────────
    #
    # Each line is:  'column_name' : value_to_put_in_that_column
    #
    # IMPORTANT NAMING CONVENTION:
    # Fields that still need cleaning are called 'field_raw' (e.g., 'gender_raw')
    # This makes it clear which values are original and which are cleaned.
    # We NEVER overwrite the original — we always create a new cleaned column.
    # Reason: if our cleaning logic has a bug, we can always go back to the original.

    return {
        # ── Unique identifier ────────────────────────────────────────────────
        '_id'                    : record.get('_id'),

        # ── Applicant personal info ──────────────────────────────────────────
        'full_name'              : ai.get('full_name'),
        'email'                  : ai.get('email'),
        'ssn'                    : ai.get('ssn'),        # Social Security Number
        'ip_address'             : ai.get('ip_address'), # IP at time of application
        'gender_raw'             : ai.get('gender'),     # RAW — inconsistent (M/Male/F/Female)
        'date_of_birth_raw'      : ai.get('date_of_birth'), # RAW — 3 different formats
        'zip_code'               : ai.get('zip_code'),

        # ── Financial info ───────────────────────────────────────────────────
        'annual_income_raw'      : fin.get('annual_income'),  # RAW — some are strings!
        'credit_history_months'  : fin.get('credit_history_months'), # some are negative!
        'debt_to_income'         : fin.get('debt_to_income'), # one is > 1.0 (impossible)
        'savings_balance'        : fin.get('savings_balance'),

        # ── Loan decision ────────────────────────────────────────────────────
        'loan_approved'          : dec.get('loan_approved'),   # True or False
        'interest_rate'          : dec.get('interest_rate'),   # only present if approved
        'approved_amount'        : dec.get('approved_amount'), # only present if approved
        'rejection_reason'       : dec.get('rejection_reason'),# only present if rejected

        # ── Derived spending features ────────────────────────────────────────
        'total_spending'         : total_spend,
        'alcohol_spend'          : alcohol_spend,   #
    }

# ─── Apply the function to every record ─────────────────────────────────────
#
# [flatten_record(r) for r in raw_data] is a "list comprehension":
# it loops over every record r in raw_data and applies our function to it.
# The result is a list of 502 flat dictionaries.
#
# pd.DataFrame([...]) takes that list of dictionaries and converts it into
# a DataFrame — a table with rows (applications) and columns (fields).

df_raw = pd.DataFrame([flatten_record(r) for r in raw_data])

# ─── Quick sanity check ───────────────────────────────────────────────────────
print(f'DataFrame created successfully!')
print(f'Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
# .shape returns (number_of_rows, number_of_columns)

print(f'\nAll column names:')
for col in df_raw.columns:
    print(f'  - {col}')

print(f'\nFirst 2 rows (preview):')
df_raw.head(2)  # .head(2) shows the first 2 rows


DataFrame created successfully!
Shape: 502 rows × 18 columns

All column names:
  - _id
  - full_name
  - email
  - ssn
  - ip_address
  - gender_raw
  - date_of_birth_raw
  - zip_code
  - annual_income_raw
  - credit_history_months
  - debt_to_income
  - savings_balance
  - loan_approved
  - interest_rate
  - approved_amount
  - rejection_reason
  - total_spending
  - alcohol_spend

First 2 rows (preview):


,_id,full_name,email,ssn,ip_address,gender_raw,date_of_birth_raw,zip_code,annual_income_raw,credit_history_months,debt_to_income,savings_balance,loan_approved,interest_rate,approved_amount,rejection_reason,total_spending,alcohol_spend
0,app_200,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,NaN,NaN,algorithm_risk_score,1517,247
1,app_037,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,NaN,NaN,algorithm_risk_score,947,0


In [345]:
#getting a quick overview of the data types and missing values in each column
df_raw.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
_id,502,500,app_042,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
full_name,502,475,Susan Flores,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
email,502,494,,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ssn,497,494,652-70-5530,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ip_address,497,496,192.168.91.142,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender_raw,501,5,Male,195,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_of_birth_raw,501,494,,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zip_code,501,196,10048,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
annual_income_raw,497.0,132.0,79000.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
credit_history_months,502.0,NaN,NaN,NaN,50.40239,31.234824,-10.0,27.25,48.0,72.0,133.0


## Section 3 — Issue #1: Duplicate Records

In [346]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4: Detect duplicate records
# ═══════════════════════════════════════════════════════════════════════════

# value_counts() counts how many times each unique value appears in a column.
id_counts = df_raw['_id'].value_counts()


# We filter the count series to only show IDs that appear more than once.
dup_ids = id_counts[id_counts > 1].index.tolist()


# Get the actual rows that are duplicated so we can inspect them
dup_records = df_raw[df_raw['_id'].isin(dup_ids)]
# df_raw['_id'].isin(dup_ids) returns True for every row whose _id is in our
# list of duplicate IDs. Wrapping df_raw[...] around it filters the table to
# only those rows.

# Print the summary
print(f'Total records in raw file    : {len(df_raw):,}')
print(f'Number of unique _id values  : {df_raw["_id"].nunique():,}')
# .nunique() = "number of unique values"

print(f'Duplicate _id values found   : {dup_ids}')
print(f'Total rows affected          : {len(dup_records)}')

# Show the duplicate pairs side by side so we can confirm they are identical
print(f'\n--- Inspecting the duplicate pairs ---')
for dup_id in dup_ids:
    # Filter to just the rows with this specific duplicate ID
    subset = df_raw[df_raw['_id'] == dup_id][
        ['_id', 'full_name', 'annual_income_raw', 'loan_approved']
    ]
    print(f'\nDuplicate: {dup_id}')
    print(subset.to_string(index=False))
    # to_string(index=False) prints the table without the row numbers on the left
    # which makes the output cleaner and easier to read


Total records in raw file    : 502
Number of unique _id values  : 500
Duplicate _id values found   : ['app_042', 'app_001']
Total rows affected          : 4

--- Inspecting the duplicate pairs ---

Duplicate: app_042
    _id    full_name annual_income_raw  loan_approved
app_042 Joseph Lopez             69000          False
app_042 Joseph Lopez             69000          False

Duplicate: app_001
    _id        full_name annual_income_raw  loan_approved
app_001 Stephanie Nguyen            102000          False
app_001 Stephanie Nguyen            102000          False


In [347]:
ssn_counts = df_raw['ssn'].value_counts()
dup_ssns = ssn_counts[ssn_counts > 1].index.tolist()

print(f'\nDuplicate SSN values found   : {dup_ssns}')
print(f'Total rows affected          : {df_raw["ssn"].isin(dup_ssns).sum()}')
mask = df_raw['ssn'].isin(dup_ssns)
dup_ssn_records = df_raw[mask][['_id', 'full_name', 'credit_history_months',  'ssn', 'annual_income_raw', 'loan_approved']]
print(f'\n--- Inspecting the duplicate SSN records ---')
for ssn in dup_ssns:
    subset = dup_ssn_records[dup_ssn_records['ssn'] == ssn]
    print(f'\nDuplicate SSN: {ssn}')
    print(subset.to_string(index=False))


Duplicate SSN values found   : ['652-70-5530', '937-72-8731', '780-24-9300']
Total rows affected          : 6

--- Inspecting the duplicate SSN records ---

Duplicate SSN: 652-70-5530
    _id    full_name  credit_history_months         ssn annual_income_raw  loan_approved
app_042 Joseph Lopez                     43 652-70-5530             69000          False
app_042 Joseph Lopez                     43 652-70-5530             69000          False

Duplicate SSN: 937-72-8731
    _id    full_name  credit_history_months         ssn annual_income_raw  loan_approved
app_101 Sandra Smith                      0 937-72-8731             55000          False
app_234  Samuel Hill                     60 937-72-8731             96000          False

Duplicate SSN: 780-24-9300
    _id      full_name  credit_history_months         ssn annual_income_raw  loan_approved
app_088 Susan Martinez                     55 780-24-9300             55000          False
app_016    Gary Wilson                     

In [348]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4B: Classify every duplicate-SSN pair and create ssn_conflict flag
# ═══════════════════════════════════════════════════════════════════════════
#
# Three cases we have seen in the data:
#
#  Case A — SAME application ID, same name
#    → This is just a duplicate record (already caught in Cell 4).
#       The SSN is fine; the row itself is the problem.
#       Action: the dedup step (Cell 5) removes the second row automatically.
#
#  Case B — DIFFERENT application ID, SAME full name
#    → The same real person submitted two separate applications.
#       A single person owning one SSN is normal.
#       Action: flag both rows so the analyst can decide whether to keep both.
#
#  Case C — DIFFERENT application ID, DIFFERENT full name
#    → Two different people claim the same SSN. That is either a data-entry
#       error or a fraud indicator. This is the most serious finding.
#       Action: flag both rows as a fraud-risk / identity-conflict.

# We will classify on df_raw (before dedup) so we catch all occurrences,
# then we apply the flag to df (after dedup) in the next step.

ssn_conflict_ids = set()   # _id values that carry a Case-B or Case-C conflict

print('SSN duplicate classification:')
print(f'  {"SSN":<15}  {"Case":<8}  {"IDs":<30}  Notes')
print('  ' + '-'*85)

for ssn in dup_ssns:
    rows = df_raw[df_raw['ssn'] == ssn]
    ids   = rows['_id'].tolist()
    names = rows['full_name'].tolist()

    same_id   = len(set(ids))   == 1   # all rows share the same application ID
    same_name = len(set(names)) == 1   # all rows share the same full name

    if same_id:
        # Case A: pure duplicate record, dedup handles it
        case  = 'A'
        notes = 'Exact duplicate row — dedup removes second occurrence'
    elif same_name:
        # Case B: same person, multiple applications
        case  = 'B'
        notes = '⚠  Same person, two applications — flagged for analyst review'
        ssn_conflict_ids.update(ids)
    else:
        # Case C: different people, same SSN — fraud / identity risk
        case  = 'C'
        notes = 'DIFFERENT people, same SSN — possible identity fraud, escalate to compliance'
        ssn_conflict_ids.update(ids)

    print(f'  {ssn:<15}  Case {case}   {str(ids):<30}  {notes}')

print(f'\nRecords flagged as ssn_conflict (Cases B + C): {len(ssn_conflict_ids)}')
print(f'IDs: {sorted(ssn_conflict_ids)}')

# ─── Add the flag to df_raw so it survives into df after dedup ───────────────────────
# We add it to df_raw now; the dedup step (Cell 5) calls .copy() so the column
# carries through into df automatically.
df_raw['ssn_conflict'] = df_raw['_id'].isin(ssn_conflict_ids)

print(f'\n✓ ssn_conflict flag added to df_raw')
print(f'  True  = this record shares its SSN with a DIFFERENT application (Cases B/C)')
print(f'  False = SSN is unique, or the duplicate is just an exact-duplicate row (Case A)')

SSN duplicate classification:
  SSN              Case      IDs                             Notes
  -------------------------------------------------------------------------------------
  652-70-5530      Case A   ['app_042', 'app_042']          Exact duplicate row — dedup removes second occurrence
  937-72-8731      Case C   ['app_101', 'app_234']          DIFFERENT people, same SSN — possible identity fraud, escalate to compliance
  780-24-9300      Case C   ['app_088', 'app_016']          DIFFERENT people, same SSN — possible identity fraud, escalate to compliance

Records flagged as ssn_conflict (Cases B + C): 4
IDs: ['app_016', 'app_088', 'app_101', 'app_234']

✓ ssn_conflict flag added to df_raw
  True  = this record shares its SSN with a DIFFERENT application (Cases B/C)
  False = SSN is unique, or the duplicate is just an exact-duplicate row (Case A)


In [349]:
df = df_raw.drop_duplicates(subset='_id', keep='first').copy()

# reset_index(drop=True) renumbers the rows from 0 to N-1.
# After dropping rows, the original row numbers (0-501) have gaps.
# Resetting gives us clean sequential numbers: 0, 1, 2, 3... 499.
# drop=True means: throw away the old index rather than saving it as a column.
df.reset_index(drop=True, inplace=True)
# inplace=True means: modify df directly instead of returning a new DataFrame

print(f'Records BEFORE deduplication: {len(df_raw):,}')
print(f'Records AFTER  deduplication: {len(df):,}')
print(f'Rows removed               : {len(df_raw) - len(df)}')
print(f'\n✓ Duplicate removal complete')




Records BEFORE deduplication: 502
Records AFTER  deduplication: 500
Rows removed               : 2

✓ Duplicate removal complete


## Section 4 — Issue #2: Inconsistent Gender Coding

In [350]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6: Visualise the raw gender values — see the problem with our own eyes
# ═══════════════════════════════════════════════════════════════════════════

# fillna('NULL') temporarily replaces actual Python None values with the
# text string 'NULL' so they show up in the chart.
# Without this, None values would be silently ignored in the count.
raw_gender_counts = df['gender_raw'].fillna('NULL').value_counts()

print('Raw gender values and their counts:')
print(raw_gender_counts.to_string())
# to_string() prints the full Series without truncation


Raw gender values and their counts:
gender_raw
Male      194
Female    193
F          58
M          53
            2


In [351]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 7: Standardise all gender values to 'Male', 'Female', or NaN
# ═══════════════════════════════════════════════════════════════════════════

# ─── Step 1: Define the mapping (translation table) ──────────────────────────
# A Python dictionary maps each possible raw value to what it should become.
# np.nan is Python's official way to represent "missing / not available".

GENDER_MAP = {
    'Male'   : 'Male',    # already correct, keep as-is
    'M'      : 'Male',    # abbreviation → full word
    'Female' : 'Female',  # already correct, keep as-is
    'F'      : 'Female',  # abbreviation → full word
    ''       : np.nan,    # empty string → unknown (NaN = Not a Number, used for missing)
    None     : np.nan,    # Python None → unknown
}

# ─── Step 2: Apply the mapping to create a new clean column ──────────────────
#
# .map(GENDER_MAP) goes through every value in the 'gender_raw' column,
# looks it up in GENDER_MAP, and returns the corresponding standardised value.
# We store the result in a NEW column called 'gender'.
# df['gender_raw'] is left completely unchanged as our audit trail.

df['gender'] = df['gender_raw'].map(GENDER_MAP)

# ─── Step 3: Verify the result ───────────────────────────────────────────────
print('=== BEFORE vs AFTER Gender Standardisation ===')
print(f'\nBEFORE (raw values):')
print(df['gender_raw'].fillna('NULL').value_counts().to_string())

print(f'\nAFTER (standardised values):')
print(df['gender'].value_counts(dropna=False).to_string())
# dropna=False means: include the NaN count in the output
# By default value_counts() hides NaN — we want to see it

# Count how many were actually changed (had abbreviated form)
changed = (df['gender_raw'].isin(['M', 'F'])).sum()
unknown = df['gender'].isna().sum()

print(f'\nSummary:')
print(f'  Records changed (M→Male or F→Female)  : {changed}')
print(f'  Records set to unknown (NaN)           : {unknown}')
print(f'  Records already correct (no change)   : {len(df) - changed - unknown}')


=== BEFORE vs AFTER Gender Standardisation ===

BEFORE (raw values):
gender_raw
Male      194
Female    193
F          58
M          53
            2

AFTER (standardised values):
gender
Female    251
Male      247
NaN         2

Summary:
  Records changed (M→Male or F→Female)  : 111
  Records set to unknown (NaN)           : 2
  Records already correct (no change)   : 387


## Section 5 — Issue #3: Annual Income Stored as Text Instead of a Number

In [352]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 8: Detect records where annual_income is stored as text (string)
# ═══════════════════════════════════════════════════════════════════════════

# ─── Check what data types are actually in the column ─────────────────────────
# .apply(type) applies Python's built-in type() function to every value.
# value_counts() then counts how many of each type there are.

type_breakdown = df['annual_income_raw'].apply(type).value_counts()
print('Data types found in annual_income_raw column:')
print(type_breakdown.to_string())

# ─── Isolate the rows where income is stored as a string ─────────────────────
# isinstance(x, str) returns True if x is a text string, False otherwise
# .apply() runs this check on every value in the column
# The result is a column of True/False values — called a "boolean mask"

string_income_mask = df['annual_income_raw'].apply(lambda x: isinstance(x, str))
# lambda x: ... means: "for each value x, run this small function"
# It is equivalent to writing:
#   def check_if_string(x):
#       return isinstance(x, str)
#   string_income_mask = df['annual_income_raw'].apply(check_if_string)
# But lambdas are more concise for simple one-line functions

string_income_rows = df[string_income_mask][['_id', 'annual_income_raw', 'loan_approved']]

print(f'\nRecords with string income: {string_income_mask.sum()}')
print(string_income_rows.to_string(index=False))


Data types found in annual_income_raw column:
annual_income_raw
<class 'int'>         486
<class 'str'>           8
<class 'NoneType'>      5
<class 'float'>         1

Records with string income: 8
    _id annual_income_raw  loan_approved
app_088             55000          False
app_135             65000          False
app_446             73000           True
app_389             51000           True
app_026             72000           True
app_312             80000           True
app_180            111000          False
app_224             93000          False


In [353]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 9: Fix the type mismatch — convert everything to a proper number
# ═══════════════════════════════════════════════════════════════════════════

# pd.to_numeric() converts values to numbers.
#
# errors='coerce' is the key argument:
#   - If a value IS a valid number (like 85000 or "85000"), convert it normally
#   - If a value CANNOT be converted (like "unknown" or None), replace it with NaN
#     instead of crashing with an error

df['annual_income'] = pd.to_numeric(df['annual_income_raw'], errors='coerce')
# We create a NEW column 'annual_income' with the clean numeric values
# 'annual_income_raw' is kept unchanged as our audit trail

# ─── Verify the fix ───────────────────────────────────────────────────────────
# Check data type of the new column
print(f'Data type of annual_income_raw : {df["annual_income_raw"].dtype}')
# 'object' means "mixed types" — pandas uses this for text/mixed columns
print(f'Data type of annual_income     : {df["annual_income"].dtype}')
# 'float64' means 64-bit floating point numbers — the correct numeric type

# Check remaining NaN values (these are the truly missing ones, not the strings)
remaining_nan = df['annual_income'].isna().sum()
print(f'\nNaN values after transformation: {remaining_nan}')
print(f'  → These are the genuinely missing incomes (the None values)')
print(f'  → We will handle them in the next section')

# Show the income range to confirm the numbers look sensible
valid_incomes = df['annual_income'].dropna()
# .dropna() removes NaN values before we compute statistics
print(f'\nIncome range (valid records):')
print(f'  Minimum : ${valid_incomes.min():,.0f}')
print(f'  Maximum : ${valid_incomes.max():,.0f}')
print(f'  Average : ${valid_incomes.mean():,.0f}')
print(f'  Median  : ${valid_incomes.median():,.0f}')




Data type of annual_income_raw : object
Data type of annual_income     : float64

NaN values after transformation: 5
  → These are the genuinely missing incomes (the None values)
  → We will handle them in the next section

Income range (valid records):
  Minimum : $0
  Maximum : $171,000
  Average : $82,694
  Median  : $81,000


## Section 6 — Issue #4: Missing Income Values (Null)
5 records have `annual_income = null` — the field is completely absent. These are not zero-income applicants (which would be stored as `0`). The data literally does not exist for these people. Thus we decided to flag thme and give them the median income.

In [354]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 10: Detect and handle missing income values
# ═══════════════════════════════════════════════════════════════════════════

# ─── Identify which rows have missing (NaN) income ───────────────────────────

null_income_mask = df['annual_income'].isna()

# Show those 5 rows
null_income_rows = df[null_income_mask][['_id', 'annual_income_raw', 'loan_approved', 'gender']]
print(f'Records with missing income: {null_income_mask.sum()}')
print(null_income_rows.to_string(index=False))

# ─── Calculate the median income to use for imputation ───────────────────────
# .median() ignores NaN values automatically — it only computes over valid numbers
# This is why we do not need to .dropna() first
income_median = df['annual_income'].median()
print(f'\nMedian income (computed from {(~null_income_mask).sum()} valid records): ${income_median:,.0f}')
# ~ is the "NOT" operator — (~null_income_mask) means "records that are NOT null"


Records with missing income: 5
    _id annual_income_raw  loan_approved gender
app_436              None           True Female
app_421              None           True   Male
app_479              None          False Female
app_463              None          False Female
app_449              None           True Female

Median income (computed from 495 valid records): $81,000


In [355]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 11: Apply median imputation + create audit flag column
# ═══════════════════════════════════════════════════════════════════════════

# ─── Step 1: Create the flag column BEFORE filling the NaN values ─────────────
# We do this first because once we fill the NaN, we lose the ability to
# detect which rows were imputed.
#
# null_income_mask is True for the 5 rows that need imputation.
# We store it as a column called 'income_imputed' — so True means "this
# income value was imputed (estimated), not measured from the application".

df['income_imputed'] = null_income_mask

# ─── Step 2: Fill NaN values with the median ─────────────────────────────────
# .fillna(value) replaces every NaN in the column with the given value.
# The median we calculated $81,000 gets placed into those 5 rows.

df['annual_income'] = df['annual_income'].fillna(income_median)

# ─── Verify: no NaN values should remain ─────────────────────────────────────
remaining_nan = df['annual_income'].isna().sum()
imputed_count = df['income_imputed'].sum()
# .sum() on a boolean column counts the True values

print(f'Remaining NaN in annual_income: {remaining_nan}  ← should be 0')
print(f'Rows flagged as imputed        : {imputed_count}  ← should be 5')
print(f'\n✓ The 5 missing incomes have been filled with median: ${income_median:,.0f}')
print(f'✓ Flag column income_imputed added — True for those 5 rows')

# Show the 5 fixed rows to confirm
print(f'\nThe 5 imputed rows after fix:')
print(df[df['income_imputed']][['_id', 'annual_income', 'income_imputed', 'loan_approved']].to_string(index=False))

Remaining NaN in annual_income: 0  ← should be 0
Rows flagged as imputed        : 5  ← should be 5

✓ The 5 missing incomes have been filled with median: $81,000
✓ Flag column income_imputed added — True for those 5 rows

The 5 imputed rows after fix:
    _id  annual_income  income_imputed  loan_approved
app_436        81000.0            True           True
app_421        81000.0            True           True
app_479        81000.0            True          False
app_463        81000.0            True          False
app_449        81000.0            True           True


## Section 7 — Issue #5: Negative Credit History Months


We take the **absolute value** — we simply remove the minus sign. So `-10` becomes `10` and `-3` becomes `3`. Both values (10 months and 3 months) are plausible credit history lengths for the types of applicants in this dataset.

In [356]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 12: Detect and fix negative credit history months
# ═══════════════════════════════════════════════════════════════════════════

# ─── Detect negative values ───────────────────────────────────────────────────
# df['credit_history_months'] < 0 creates a True/False mask
# True = this row has a negative value (a problem)
# False = this row has a zero or positive value (OK)

neg_credit_mask = df['credit_history_months'] < 0

neg_credit_rows = df[neg_credit_mask][
    ['_id', 'credit_history_months', 'annual_income', 'loan_approved']
]

print(f'Records with negative credit_history_months: {neg_credit_mask.sum()}')
print(neg_credit_rows.to_string(index=False))

# Show the full distribution to understand the context
print(f'\nFull distribution of credit_history_months:')
print(f'  Minimum   : {df["credit_history_months"].min()} months  ← the problematic values')
print(f'  Median    : {df["credit_history_months"].median()} months')
print(f'  Maximum   : {df["credit_history_months"].max()} months')
print(f'  Mean      : {df["credit_history_months"].mean():.1f} months')

Records with negative credit_history_months: 2
    _id  credit_history_months  annual_income  loan_approved
app_043                    -10       131000.0           True
app_156                     -3        25000.0          False

Full distribution of credit_history_months:
  Minimum   : -10 months  ← the problematic values
  Median    : 48.5 months
  Maximum   : 133 months
  Mean      : 50.4 months


In [357]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 13: Fix negative values by taking the absolute value
# ═══════════════════════════════════════════════════════════════════════════

# ─── Step 1: Add a flag column before making any changes ─────────────────────
# Same principle as with income: record which rows were modified
# so downstream analysts know this value was corrected
df['credit_history_months_flag'] = neg_credit_mask

# ─── Step 2: Apply abs() — take the absolute value ───────────────────────────
# .abs() is a pandas method that removes the minus sign from all negative numbers
# Positive numbers are unchanged: abs(48) = 48
# Negative numbers become positive: abs(-10) = 10
df['credit_history_months'] = df['credit_history_months'].abs()
# Note: we modify this column in-place because there is no separate _raw version
# needed here — the correction is straightforward (just remove the minus sign)

# ─── Verify ───────────────────────────────────────────────────────────────────
print('Fixed rows:')
print(df[df['credit_history_months_flag']][
    ['_id', 'credit_history_months', 'credit_history_months_flag']
].to_string(index=False))

print(f'\nNew minimum credit_history_months: {df["credit_history_months"].min()} months  ← no longer negative')
print(f'✓ All credit history values are now non-negative')

Fixed rows:
    _id  credit_history_months  credit_history_months_flag
app_043                     10                        True
app_156                      3                        True

New minimum credit_history_months: 0 months  ← no longer negative
✓ All credit history values are now non-negative


## Section 8 — Issue #6: Impossible Debt-to-Income Ratio
The most likely explanation is a **decimal point error** — the value should be `0.185` (18.5%), not `1.85` (185%). A DTI of 18.5% is completely normal and consistent with the rest of the dataset's distribution.

In [358]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 14: Detect and fix the impossible DTI value
# ═══════════════════════════════════════════════════════════════════════════

# ─── Detect values outside the valid [0, 1] range ────────────────────────────
# We check for TWO impossible conditions using | (OR):
#   1. Greater than 1.0 (more than 100% of income in debt)
#   2. Less than 0 (negative debt — nonsensical)

bad_dti_mask = (df['debt_to_income'] > 1.0) | (df['debt_to_income'] < 0)

bad_dti_rows = df[bad_dti_mask][['_id', 'debt_to_income', 'annual_income', 'loan_approved']]

print(f'Records with DTI outside valid [0, 1] range: {bad_dti_mask.sum()}')
print(bad_dti_rows.to_string(index=False))

# Show the distribution of valid DTI values for comparison
valid_dti = df.loc[~bad_dti_mask, 'debt_to_income']
# ~ is NOT — so ~bad_dti_mask means "rows that do NOT have bad DTI"
# df.loc[condition, column] selects rows matching the condition, then the column
print(f'\nDTI distribution (valid records only):')
print(f'  Min    : {valid_dti.min():.3f}')
print(f'  Median : {valid_dti.median():.3f}')
print(f'  Max    : {valid_dti.max():.3f}')
print(f'  Mean   : {valid_dti.mean():.3f}')
print(f'\nFor reference: 1.85 / 10 = {1.85/10:.3f} ← consistent with valid range')


Records with DTI outside valid [0, 1] range: 1
    _id  debt_to_income  annual_income  loan_approved
app_402            1.85        88000.0           True

DTI distribution (valid records only):
  Min    : 0.050
  Median : 0.230
  Max    : 0.450
  Mean   : 0.242

For reference: 1.85 / 10 = 0.185 ← consistent with valid range


In [359]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 15: Apply the decimal point correction
# ═══════════════════════════════════════════════════════════════════════════

# Add audit flag first
df['dti_flag'] = bad_dti_mask

# Fix: divide only the problematic rows by 10
# df.loc[condition, column] = value  →  set a specific subset of cells to a new value
# We only modify rows where bad_dti_mask is True — all other rows are unchanged
df.loc[bad_dti_mask, 'debt_to_income'] = df.loc[bad_dti_mask, 'debt_to_income'] / 10

# Verify
print(f'Fixed row:')
print(df[df['dti_flag']][['_id', 'debt_to_income', 'dti_flag']].to_string(index=False))
print(f'✓ All DTI values now within valid [0, 1] range')


Fixed row:
    _id  debt_to_income  dti_flag
app_402           0.185      True
✓ All DTI values now within valid [0, 1] range


## Section 9 — Issue #7: Different Date Formats

The `date_of_birth` field contains dates written in **different formats** & some missing Values:





In [360]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 16: Classify what format each date string is using
# ═══════════════════════════════════════════════════════════════════════════

# re.match() checks if a string matches a pattern
# The patterns use "regular expressions" — a mini-language for describing text patterns:
#   \d    = any single digit (0–9)
#   \d{4} = exactly four consecutive digits
#   \d{2} = exactly two consecutive digits
#   -     = a literal hyphen character
#   /     = a literal slash character
#   ^     = start of string
#   $     = end of string
# So '^\d{4}-\d{2}-\d{2}$' means: "exactly 4 digits, hyphen, 2 digits, hyphen, 2 digits"

def classify_date_format(date_string):
    """
    Looks at a date string and identifies which format it uses.
    Returns a label string describing the format.

    KEY CHALLENGE — DD/MM/YYYY vs MM/DD/YYYY:
    Both formats share the same regex pattern (\d{2}/\d{2}/\d{4}).
    We resolve the ambiguity by inspecting BOTH the first and second numbers:

    Rule 1 — second part > 12:
        A month can only be 1–12, so if the SECOND number is 13 or higher
        it CANNOT be a month → the second part must be the DAY
        → format is MM/DD/YYYY
        Example: "03/20/1968" → month=3, day=20 → March 20, 1968 ✓

    Rule 2 — first part > 12 (and second part ≤ 12):
        A month can only be 1–12, so if the FIRST number is 13 or higher
        it CANNOT be a month → the first part must be the DAY
        → format is DD/MM/YYYY
        Example: "14/02/1982" → day=14, month=2 → February 14, 1982 ✓

    Rule 3 — both ≤ 12 (ambiguous):
        Either interpretation is mathematically valid.
        We cannot tell from the number alone.
        → We assume DD/MM/YYYY (European convention — NovaSBE is in Lisbon).
        Example: "03/10/1981" → we read it as October 3, 1981
        
    """
    if not date_string:               # covers None and empty string ''
        return 'MISSING'
    d = str(date_string)              # ensure it is definitely a string
    if re.match(r'^\d{4}-\d{2}-\d{2}$', d):   # e.g. "1990-03-15"
        return 'YYYY-MM-DD'
    elif re.match(r'^\d{4}/\d{2}/\d{2}$', d): # e.g. "1990/03/15"
        return 'YYYY/MM/DD'
    elif re.match(r'^\d{2}/\d{2}/\d{4}$', d): # e.g. "15/03/1990" or "03/20/1968"
        # .split('/') breaks the string on slashes: '03/20/1968' → ['03', '20', '1968']
        # int() converts the text fragment to a number so we can compare it
        parts      = d.split('/')
        first_part  = int(parts[0])   # could be a day OR a month
        second_part = int(parts[1])   # could be a day OR a month

        if second_part > 12:
            # Second part is > 12 → it CANNOT be a month → it must be the day
            # → first part is the month → MM/DD/YYYY
            return 'MM/DD/YYYY'
        elif first_part > 12:
            # First part is > 12 → it CANNOT be a month → it must be the day
            # → second part is the month → DD/MM/YYYY
            return 'DD/MM/YYYY'
        else:
            # Both parts are ≤ 12 → genuinely ambiguous
            # Assume DD/MM/YYYY (European convention)
            return 'DD/MM/YYYY'
    else:
        return f'UNKNOWN: {d}'        # catch-all for anything unexpected

# Apply the classification function to every row
df['dob_format'] = df['date_of_birth_raw'].apply(classify_date_format)
format_counts = df['dob_format'].value_counts()

print('Date format breakdown in the raw data:')
print(format_counts.to_string())
print(f'\nRecords needing normalisation (not already YYYY-MM-DD): {(df["dob_format"] != "YYYY-MM-DD").sum()}')
print(f'That is {(df["dob_format"] != "YYYY-MM-DD").mean()*100:.1f}% of all records')


Date format breakdown in the raw data:
dob_format
YYYY-MM-DD    339
DD/MM/YYYY     75
YYYY/MM/DD     56
MM/DD/YYYY     26
MISSING         4

Records needing normalisation (not already YYYY-MM-DD): 161
That is 32.2% of all records


In [361]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 17: Parse all dates into a single standard format, then derive age
# ═══════════════════════════════════════════════════════════════════════════

def parse_date_of_birth(raw_string):
    """
    Parses a date string into a Python datetime object, handling all four
    formats found in the dataset.
    Returns pd.NaT if the value is missing or cannot be parsed.

    pd.NaT = "Not a Time" — pandas' equivalent of NaN for dates.
    It behaves like a missing value in all date operations.

    Parameters
    ----------
    raw_string : str or None
        The raw date string from the JSON file.

    Returns
    -------
    datetime or pd.NaT
    """
    if not raw_string:       # handles None and empty string
        return pd.NaT

    s = str(raw_string)

    # ── Case 1: YYYY-MM-DD  e.g. "1990-03-15" ────────────────────────────────
    if re.match(r'^\d{4}-\d{2}-\d{2}$', s):
        try:
            return datetime.strptime(s, '%Y-%m-%d')
        except ValueError:
            return pd.NaT

    # ── Case 2: YYYY/MM/DD  e.g. "1990/03/15" ────────────────────────────────
    if re.match(r'^\d{4}/\d{2}/\d{2}$', s):
        try:
            return datetime.strptime(s, '%Y/%m/%d')
        except ValueError:
            return pd.NaT

    # ── Case 3 & 4: DD/MM/YYYY or MM/DD/YYYY  e.g. "14/02/1982" or "03/20/1968"
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s):
        parts       = s.split('/')
        first_part  = int(parts[0])   # could be day OR month
        second_part = int(parts[1])   # could be day OR month

        # Three-rule disambiguation (mirrors classify_date_format exactly):
        #
        # RULE 1 — second_part > 12:
        #   A month can never be > 12.  If the SECOND number is 13+, it CANNOT
        #   be a month → it must be the DAY → first part is the MONTH.
        #   Format: MM/DD/YYYY
        #   Example: "03/20/1968" → second=20 can't be a month → March 20, 1968 
        #
        # RULE 2 — first_part > 12 (and second_part ≤ 12):
        #   A month can never be > 12.  If the FIRST number is 13+, it CANNOT
        #   be a month → it must be the DAY → second part is the MONTH.
        #   Format: DD/MM/YYYY
        #   Example: "14/02/1982" → first=14 can't be a month → February 14, 1982 
        #
        # RULE 3 — both ≤ 12 (genuinely ambiguous):
        #   Either interpretation is mathematically possible.
        #   We assume DD/MM/YYYY — European convention (NovaSBE is in Lisbon).
        #   Example: "03/10/1981" → October 3, 1981

        if second_part > 12:
            fmt = '%m/%d/%Y'   # second is day → first is month  → MM/DD/YYYY
        elif first_part > 12:
            fmt = '%d/%m/%Y'   # first is day  → second is month → DD/MM/YYYY
        else:
            fmt = '%d/%m/%Y'   # ambiguous → European convention → DD/MM/YYYY

        # datetime.strptime(string, format_string) converts text → datetime object.
        # Format codes:  %Y = 4-digit year   %m = 2-digit month   %d = 2-digit day
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            return pd.NaT

    # If nothing matched, return NaT (missing date)
    return pd.NaT

# ─── Apply the parser to every row ───────────────────────────────────────────
df['date_of_birth'] = df['date_of_birth_raw'].apply(parse_date_of_birth)

# ─── Derive age from date of birth ───────────────────────────────────────────
# For each valid date of birth we calculate:
#   age = (AUDIT_DATE - date_of_birth).days ÷ 365
#
# (AUDIT_DATE - date) produces a timedelta (a duration between two dates).
# .days converts that duration to an integer number of days.
# // 365 performs integer (floor) division to get whole years — no rounding up.
#
# For rows where date_of_birth is NaT we return np.nan (no date = no age).

df['age'] = df['date_of_birth'].apply(
    lambda d: (AUDIT_DATE - d).days // 365 if pd.notna(d) else np.nan
)
# pd.notna(d) returns True if d is a real datetime, False if it is NaT

# ─── Verify ───────────────────────────────────────────────────────────────────
successfully_parsed = df['date_of_birth'].notna().sum()
failed_to_parse     = df['date_of_birth'].isna().sum()

print(f'Dates successfully parsed : {successfully_parsed}  ← should be 496 (500 − 4 truly null)')
print(f'Could not parse (NaT)     : {failed_to_parse}  ← should be 4 (empty string / None in original JSON)')
print(f'\nAge statistics:')
print(f'  Youngest applicant : {df["age"].min():.0f} years old')
print(f'  Oldest applicant   : {df["age"].max():.0f} years old')
print(f'  Average age        : {df["age"].mean():.1f} years')
print(f'  Median age         : {df["age"].median():.1f} years')



Dates successfully parsed : 496  ← should be 496 (500 − 4 truly null)
Could not parse (NaT)     : 4  ← should be 4 (empty string / None in original JSON)

Age statistics:
  Youngest applicant : 23 years old
  Oldest applicant   : 67 years old
  Average age        : 40.8 years
  Median age         : 39.0 years


## Section 10 — Issue #8: Missing Date-of-Birth Values

Now that all four date formats have been normalised in Section 9, we can clearly see which records are **completely missing** a date of birth — the raw field was `null` (Python `None`) in the original JSON.

The `parse_date_of_birth` function in Cell 17 already handled this: whenever it received `None` or an empty string it immediately returned `pd.NaT` (**Not a Time**) — pandas' special placeholder for a missing date, equivalent to `NaN` for numbers.

As a result, those **4 records** currently have:
- `date_of_birth` = `NaT` — no date available
- `age` = `NaN` — impossible to compute age without a birth date

**Why we do NOT impute a date of birth:**  
For income we filled in the median — that was acceptable because income is a financial estimate. Date of birth is **PII — Personally Identifiable Information**. Fabricating it would mean inventing sensitive personal data about a real person, which violates data governance principles and privacy regulations such as **GDPR**. We must leave the gap and document it honestly.

**Our strategy — same two-step approach as every other issue:**
1. **Detect** — identify the exact rows affected using `.isna()`
2. **Flag** — add a `dob_missing` column (`True` = missing) so downstream analysts know at a glance which rows have no birth date

In [362]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 18: Detect records with a missing date of birth
# ═══════════════════════════════════════════════════════════════════════════

# ─── Build a boolean mask for missing dates ───────────────────────────────────
# df['date_of_birth'].isna() scans every value in the column and returns
# True  wherever the value is NaT (missing date)
# False wherever the value is a real parsed datetime
#
# .isna() works for BOTH numeric missing values (NaN) AND date missing values (NaT)
# It is the standard pandas way to detect any kind of "nothing here" value.
#
# Compare with how we detected missing income in Cell 10:
#   null_income_mask = df['annual_income'].isna()   ← same pattern, different column

missing_dob_mask = df['date_of_birth'].isna()
# True  = this applicant has no date of birth in the raw data
# False = this applicant has a successfully parsed date of birth

# ─── Isolate the affected rows so we can inspect them ─────────────────────────
# We select a small set of informative columns so the output is easy to read:
#   _id              — so we know exactly which application is affected
#   date_of_birth_raw — the original value from the JSON (should be None)
#   date_of_birth    — the parsed value (will show NaT)
#   age              — the derived value (will show NaN)

missing_dob_rows = df[missing_dob_mask][
    ['_id', 'date_of_birth_raw', 'date_of_birth', 'age']
]

print(f'Total records in dataset          : {len(df):,}')
print(f'Records WITH a valid date of birth: {(~missing_dob_mask).sum():,}')
# ~ is the NOT operator — (~missing_dob_mask) flips True→False and False→True
# so it selects all rows that DO have a date

print(f'Records with MISSING date of birth: {missing_dob_mask.sum()}')
print(f'\nThe affected rows:')
print(missing_dob_rows.to_string(index=False))
# to_string(index=False) prints the table without the internal row numbers on the left

print(f'\nNote: date_of_birth_raw is None for all of them')
print(f'      → the field was completely absent in the original JSON file')
print(f'      → this is a Completeness issue — the data simply does not exist')




Total records in dataset          : 500
Records WITH a valid date of birth: 496
Records with MISSING date of birth: 4

The affected rows:
    _id date_of_birth_raw date_of_birth  age
app_075                             NaT  NaN
app_120                             NaT  NaN
app_350                             NaT  NaN
app_165                             NaT  NaN

Note: date_of_birth_raw is None for all of them
      → the field was completely absent in the original JSON file
      → this is a Completeness issue — the data simply does not exist


In [363]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 19: Flag the missing dates — do NOT impute
# ═══════════════════════════════════════════════════════════════════════════

# ─── Step 1: Create the audit flag column ────────────────────────────────────
#   Here:    df['dob_missing']                 = missing_dob_mask
#
# We ALWAYS record which rows were affected BEFORE touching anything.
# True  = this applicant's date of birth was absent in the raw JSON
# False = this applicant has a valid, parsed date of birth

df['dob_missing'] = missing_dob_mask
# True for the rows with no date, False for everyone else

# ─── Step 2: Verify nothing was changed unintentionally ──────────────────────
print(f'Flag column dob_missing added successfully')
print(f'  Rows where dob_missing = True  : {df["dob_missing"].sum()}')
print(f'  Rows where dob_missing = False : {(~df["dob_missing"]).sum()}')
# These two numbers must add up to {len(df)} (total records after deduplication)

print(f'\nNaT still in date_of_birth column : {df["date_of_birth"].isna().sum()}  ← intentionally kept')
print(f'NaN still in age column           : {df["age"].isna().sum()}  ← intentionally kept')

# ─── Step 3: Show the flagged rows for final confirmation ────────────────────
print(f'\nFlagged rows — date_of_birth and age deliberately left blank:')
print(
    df[df['dob_missing']][['_id', 'date_of_birth', 'age', 'dob_missing']]
    .to_string(index=False)
)

print(f'\n✓ Flag column dob_missing added')
print(f'✓ date_of_birth and age remain NaT / NaN — no personal info invented')
print(f'✓ Governance-safe: these records are transparently flagged for downstream use')


Flag column dob_missing added successfully
  Rows where dob_missing = True  : 4
  Rows where dob_missing = False : 496

NaT still in date_of_birth column : 4  ← intentionally kept
NaN still in age column           : 4  ← intentionally kept

Flagged rows — date_of_birth and age deliberately left blank:
    _id date_of_birth  age  dob_missing
app_075           NaT  NaN         True
app_120           NaT  NaN         True
app_350           NaT  NaN         True
app_165           NaT  NaN         True

✓ Flag column dob_missing added
✓ date_of_birth and age remain NaT / NaN — no personal info invented
✓ Governance-safe: these records are transparently flagged for downstream use


## Section 11 — Issue #9: Missing Email and SSN (Personal Identifying Information)

In [364]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 20: Detect missing email and SSN fields
# ═══════════════════════════════════════════════════════════════════════════

# ─── Email check ─────────────────────────────────────────────────────────────
# We need to catch THREE cases where email is functionally "missing":
#   1. The value is '' (empty string)
#   2. The value is None (Python null)
#   3. The value is NaN (pandas null)
#
# .isin(['', None]) returns True for empty string or Python None
# .isna() returns True for NaN or None
# | is the OR operator — True if EITHER condition is True

missing_email_mask = df['email'].isin(['', None]) | df['email'].isna()

# ─── SSN check ───────────────────────────────────────────────────────────────
# SSN only has NaN/None (no empty string problem here)
missing_ssn_mask = df['ssn'].isna()

# ─── Show the affected records ───────────────────────────────────────────────
missing_email_rows = df[missing_email_mask][['_id', 'email', 'loan_approved']]
missing_ssn_rows   = df[missing_ssn_mask][['_id', 'ssn', 'loan_approved']]

print(f'Records with missing email: {missing_email_mask.sum()}')
print(missing_email_rows.to_string(index=False))

print(f'\nRecords with missing SSN: {missing_ssn_mask.sum()}')
print(missing_ssn_rows.to_string(index=False))


Records with missing email: 7
    _id email  loan_approved
app_075                 True
app_413                 True
app_120                False
app_268                 True
app_377                 True
app_350                 True
app_165                 True

Records with missing SSN: 4
    _id ssn  loan_approved
app_075 NaN           True
app_120 NaN          False
app_268 NaN           True
app_165 NaN           True


In [365]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 21: Add audit flags — DO NOT impute PII fields
# ═══════════════════════════════════════════════════════════════════════════

# Add boolean flag columns
# True = this record is missing this PII field
df['email_missing'] = missing_email_mask
df['ssn_missing']   = missing_ssn_mask

print(f'✓ email_missing flag column added ({missing_email_mask.sum()} True values)')
print(f'✓ ssn_missing flag column added   ({missing_ssn_mask.sum()} True values)')
print()
print('These records have been flagged for review by the Governance Officer.')
print('No email or SSN values have been fabricated.')
print()

# Check if any of these missing-PII records were approved for loans
email_approved = df[missing_email_mask]['loan_approved'].sum()
ssn_approved   = df[missing_ssn_mask]['loan_approved'].sum()
print(f'Of the {missing_email_mask.sum()} records with missing email : {email_approved} were approved for loans')
print(f'Of the {missing_ssn_mask.sum()} records with missing SSN   : {ssn_approved} were approved for loans')
print(f'\n✓ Governance-safe: missing PII records are transparently flagged for downstream use, Clear compliance risks without fabricating any personal information')


✓ email_missing flag column added (7 True values)
✓ ssn_missing flag column added   (4 True values)

These records have been flagged for review by the Governance Officer.
No email or SSN values have been fabricated.

Of the 7 records with missing email : 6 were approved for loans
Of the 4 records with missing SSN   : 3 were approved for loans

✓ Governance-safe: missing PII records are transparently flagged for downstream use, Clear compliance risks without fabricating any personal information


## Section 12 — Full Data Quality Dashboard

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 22: Build a live-computed summary dashboard of ALL 11 data quality issues
# ═══════════════════════════════════════════════════════════════════════════
#
# Every count is computed live from the DataFrame — nothing is hard-coded.
# This means the table is always correct even if the underlying data changes.
#
# We build the table as a list of dicts (one per row), then convert to a
# DataFrame for clean tabular display.

total_raw    = len(df_raw)   # 502 — original record count before dedup
total_clean  = len(df)       # 500 — records after removing duplicate _id rows

# ─── Compute each count from the live DataFrame ————————————————————————─
# Issue #1: duplicate _id rows removed during dedup
n_dup_id           = total_raw - total_clean

# Issue #2: gender abbreviations (M or F) that were mapped to full words
n_gender           = int(df['gender_raw'].isin(['M', 'F']).sum())

# Issue #3: income values that arrived as text strings (not numeric)
#   string_income_mask was created in Cell 8 — we reuse it here
n_string_income    = int(string_income_mask.sum())

# Issue #4: income values that were missing (null) and had to be imputed
n_income_imputed   = int(df['income_imputed'].sum())

# Issue #5: credit history months that were negative (abs() applied)
n_neg_credit       = int(df['credit_history_months_flag'].sum())

# Issue #6: DTI values outside valid [0, 1] range (divided by 10)
n_bad_dti          = int(df['dti_flag'].sum())

# Issue #7: date-of-birth strings that were NOT already in YYYY-MM-DD format
#   dob_format column was created in Cell 16
n_date_format      = int((df['dob_format'] != 'YYYY-MM-DD').sum())
#   Subtract MISSING rows because they are a separate issue (#8)
n_date_format     -= int(df['dob_missing'].sum())

# Issue #8: completely missing date of birth (null / NaT in raw data)
n_dob_missing      = int(df['dob_missing'].sum())

# Issue #9: missing email address
n_email_missing    = int(df['email_missing'].sum())

# Issue #10: missing SSN
n_ssn_missing      = int(df['ssn_missing'].sum())

# Issue #11: SSN shared across DIFFERENT applications (Cases B and C)
#   ssn_conflict column was created in Cell 4B
n_ssn_conflict     = int(df['ssn_conflict'].sum())

# ─── Helper: format count as percentage of the clean dataset ——————————─
def pct(n, base=total_clean):
    """Returns a formatted percentage string, e.g. pct(5) → '1.0%'"""
    return f'{n / base * 100:.1f}%'

# ─── Build the summary table ——————————————————————————————————————─
summary_data = [
    {
        'Issue'     : '#1 — Duplicate Records (_id)',
        'Count'     : n_dup_id,
        'Pct'       : f'{n_dup_id / total_raw * 100:.1f}%',   # % of ORIGINAL 502
        'Dimension' : 'Uniqueness',
        'Severity'  : '⚠️  Medium',
        'Fix Applied': 'Removed 2nd occurrence (keep=first)',
    },
    {
        'Issue'     : '#2 — Inconsistent Gender Coding',
        'Count'     : n_gender,
        'Pct'       : pct(n_gender),
        'Dimension' : 'Consistency',
        'Severity'  : '🟡 Low',
        'Fix Applied': 'M→Male, F→Female via lookup dict',
    },
    {
        'Issue'     : '#3 — Income Stored as Text',
        'Count'     : n_string_income,
        'Pct'       : pct(n_string_income),
        'Dimension' : 'Validity / Type',
        'Severity'  : '⚠️  Medium',
        'Fix Applied': 'pd.to_numeric(errors="coerce")',
    },
    {
        'Issue'     : '#4 — Missing Income (null)',
        'Count'     : n_income_imputed,
        'Pct'       : pct(n_income_imputed),
        'Dimension' : 'Completeness',
        'Severity'  : '⚠️  Medium',
        'Fix Applied': 'Median imputation + income_imputed flag',
    },
    {
        'Issue'     : '#5 — Negative Credit History',
        'Count'     : n_neg_credit,
        'Pct'       : pct(n_neg_credit),
        'Dimension' : 'Validity',
        'Severity'  : '🟡 Low',
        'Fix Applied': 'abs() applied (−10 → 10)',
    },
    {
        'Issue'     : '#6 — DTI Ratio > 1.0',
        'Count'     : n_bad_dti,
        'Pct'       : pct(n_bad_dti),
        'Dimension' : 'Validity',
        'Severity'  : '⚠️  Medium',
        'Fix Applied': 'Divided by 10  (1.85 → 0.185)',
    },
    {
        'Issue'     : '#7 — Mixed Date Formats',
        'Count'     : n_date_format,
        'Pct'       : pct(n_date_format),
        'Dimension' : 'Consistency',
        'Severity'  : '⚠️  Medium',
        'Fix Applied': 'Multi-format parser → ISO 8601 (YYYY-MM-DD)',
    },
    {
        'Issue'     : '#8 — Missing Date of Birth',
        'Count'     : n_dob_missing,
        'Pct'       : pct(n_dob_missing),
        'Dimension' : 'Completeness',
        'Severity'  : '⚠️  Medium',
        'Fix Applied': 'Flag only — PII, no imputation (GDPR)',
    },
    {
        'Issue'     : '#9 — Missing Email Address',
        'Count'     : n_email_missing,
        'Pct'       : pct(n_email_missing),
        'Dimension' : 'Completeness',
        'Severity'  : '⚠️  Medium',
        'Fix Applied': 'Flag only — PII, no fabrication',
    },
    {
        'Issue'     : '#10 — Missing SSN',
        'Count'     : n_ssn_missing,
        'Pct'       : pct(n_ssn_missing),
        'Dimension' : 'Completeness',
        'Severity'  : '🔴 High',
        'Fix Applied': 'Flag only — PII, no fabrication',
    },
    {
        'Issue'     : '#11 — Conflicting / Duplicate SSN',
        'Count'     : n_ssn_conflict,
        'Pct'       : pct(n_ssn_conflict),
        'Dimension' : 'Uniqueness',
        'Severity'  : '🔴 High (possible fraud)',
        'Fix Applied': 'Classified A/B/C + ssn_conflict flag; B/C escalated',
    },
]

summary_df = pd.DataFrame(summary_data)

# Display settings — allow wider text so the Fix column is not truncated
pd.set_option('display.max_colwidth', 55)
pd.set_option('display.width', 130)

total_affected = sum(r['Count'] for r in summary_data)

print('\u2554' + '═'*78 + '\u2557')
print(f'\u2551  NOVACRED DATA QUALITY DASHBOARD — {len(summary_data)} ISSUES ACROSS {total_raw:,} RAW RECORDS  \u2551')
print('\u255a' + '═'*78 + '\u255d')
print()
print(summary_df.to_string(index=False))
print()
print(f'Total distinct data points affected : {total_affected:,}  (counts overlap — one record can have multiple issues)')
print(f'Records in cleaned output            : {total_clean:,}')
print(f'Records removed (duplicate _id)     : {n_dup_id}')


╔══════════════════════════════════════════════════════════════════════╗
║           NOVACRED DATA QUALITY SUMMARY — 9 ISSUES FOUND             ║
╚══════════════════════════════════════════════════════════════════════╝
                              Issue  Count   Pct       Dimension                                                 Fix
             #1 — Duplicate Records      2  0.4%      Uniqueness                 Removed 2nd occurrence (keep=first)
    #2 — Inconsistent Gender Coding    111 22.2%     Consistency       Mapped M→Male, F→Female via lookup dictionary
         #3 — Income Stored as Text      8  1.6% Validity / Type pd.to_numeric(errors="coerce") converts text→number
         #4 — Missing Income (null)      5  1.0%    Completeness             Median imputation + income_imputed flag
       #5 — Negative Credit History      2  0.4%        Validity         abs() removes the minus sign (-10→10, -3→3)
               #6 — DTI Ratio > 1.0      1  0.2%        Validity     Divided by

## Section 13 — Export: Save the Clean Dataset and Quality Report

###  `cleaned_credit_applications.csv`
A flat CSV (Comma-Separated Values) file containing all 500 records with:
- **Clean columns** (e.g. `gender`, `annual_income`, `date_of_birth`) — the fixed versions used for analysis
- **Raw columns** (e.g. `gender_raw`, `annual_income_raw`, `date_of_birth_raw`) — the original unmodified values as an audit trail
- **Flag columns** (e.g. `income_imputed`, `dti_flag`, `dob_missing`) — True/False indicators marking which values were patched or are missing

This file is the input for **Notebook 02** (Data Scientist — bias analysis) and **Notebook 03** (Governance Officer — privacy audit).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 23: Visualise the data quality dashboard as a horizontal bar chart
# ═══════════════════════════════════════════════════════════════════════════
#
# A picture is worth a thousand numbers. The bar chart lets the reader
# immediately see:
#   • Which issues affect the MOST records (mixed date formats tops the chart)
#   • Which issues are rare single-row anomalies (DTI error, negative credit)
#   • How severity (colour) correlates with count
#
# Colour coding (matches the Severity column in the dashboard table above):
#   🔴 Red    — High severity  (missing SSN, SSN conflicts / fraud risk)
#   ⚠️  Orange — Medium severity (most issues)
#   🟡 Yellow — Low severity   (gender abbreviations, negative credit)

import matplotlib.patches as mpatches   # used to build the colour legend
import os                               # used to create the figures directory

# ─── Data: pull counts and labels directly from the summary_df built in Cell 22 ─
# Reverse the order so the LARGEST bar appears at the top of the chart
labels = summary_df['Issue'].tolist()[::-1]
counts = summary_df['Count'].tolist()[::-1]

# ─── Assign a bar colour per issue based on severity ————————————————─
# We map each issue index (0 = #1, 10 = #11) to a hex colour.
# After reversing the list above, index 0 = issue #11 and index 10 = issue #1.
# So we define colours in original order (#1→#11) and reverse at the end.
COLOUR_HIGH   = '#e05c5c'   # muted red — High severity
COLOUR_MEDIUM = '#f0a04b'   # warm orange — Medium severity
COLOUR_LOW    = '#f5d76e'   # soft yellow — Low severity

# Severity assignment in issue order (#1 through #11):
colours_ordered = [
    COLOUR_MEDIUM,   # #1  Duplicate _id          — Medium
    COLOUR_LOW,      # #2  Gender coding           — Low
    COLOUR_MEDIUM,   # #3  Income as text          — Medium
    COLOUR_MEDIUM,   # #4  Missing income          — Medium
    COLOUR_LOW,      # #5  Negative credit history — Low
    COLOUR_MEDIUM,   # #6  Bad DTI                 — Medium
    COLOUR_MEDIUM,   # #7  Mixed date formats      — Medium
    COLOUR_MEDIUM,   # #8  Missing DOB             — Medium
    COLOUR_MEDIUM,   # #9  Missing email           — Medium
    COLOUR_HIGH,     # #10 Missing SSN             — High
    COLOUR_HIGH,     # #11 Conflicting SSN         — High (fraud risk)
]

# Reverse to match the reversed labels/counts lists (so colours stay paired)
colours = colours_ordered[::-1]

# ─── Build the chart —————————————————————————————————————————————─
fig, ax = plt.subplots(figsize=(12, 7))
# fig = the whole figure (the canvas)
# ax  = the axes object (the actual plot area inside the canvas)
# figsize=(width_inches, height_inches) — taller so labels do not overlap

# Draw horizontal bars
# barh(y_positions, widths, colour_list, ...) where:
#   range(len(labels)) = evenly spaced Y positions 0, 1, 2 … 10
#   counts             = the width of each bar (number of records affected)
#   color=colours      = one colour per bar from our list above
bars = ax.barh(
    range(len(labels)),
    counts,
    color=colours,
    edgecolor='white',     # thin white border between adjacent bars
    linewidth=0.8,
    height=0.65,           # bar height (fraction of available space)
)

# ─── Add count labels at the end of each bar ───────────────────────────────────────
for bar, count in zip(bars, counts):
    # bar.get_width()  — the right edge of the bar (= the count value)
    # bar.get_y() + bar.get_height()/2  — vertical centre of the bar
    ax.text(
        bar.get_width() + 1,           # x position: just past the bar tip
        bar.get_y() + bar.get_height() / 2,  # y position: vertical centre
        str(count),                    # the text to display
        va='center',                   # vertical alignment: centre on the bar
        ha='left',                     # horizontal alignment: left-justify
        fontsize=10,
        fontweight='bold',
        color='#333333',
    )

# ─── Axis labels and title ──────────────────────────────────────────────────
ax.set_yticks(range(len(labels)))
# Set the y-axis tick positions to match our 11 bars

ax.set_yticklabels(labels, fontsize=10)
# Replace the numeric tick labels (0–10) with our descriptive issue names

ax.set_xlabel('Number of Records Affected', fontsize=11)
ax.set_title(
    'NovaCred Data Quality Audit — Records Affected per Issue\n'
    '(out of 500 cleaned records; issue #1 measured against 502 raw records)',
    fontsize=13, fontweight='bold', pad=14
)

# Add a light vertical grid so the reader can gauge bar length
ax.xaxis.grid(True, linestyle='--', alpha=0.6, color='#cccccc')
ax.set_axisbelow(True)   # grid lines sit behind the bars, not on top

# Remove the top and right spines (borders) for a cleaner look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ─── Legend ──────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color=COLOUR_HIGH,   label='🔴  High severity  (missing SSN, identity/fraud conflict)'),
    mpatches.Patch(color=COLOUR_MEDIUM, label='⚠️   Medium severity (type errors, mixed formats, missing PII)'),
    mpatches.Patch(color=COLOUR_LOW,    label='🟡  Low severity   (abbreviation fixes, single-value corrections)'),
]
ax.legend(
    handles=legend_handles,
    loc='lower right',
    fontsize=9,
    framealpha=0.85,
    edgecolor='#cccccc',
)

plt.tight_layout()
# tight_layout() automatically adjusts margins so nothing is clipped

# ─── Save the figure to disk ───────────────────────────────────────────────────
CHART_PATH = FIGURES_DIR + 'data_quality_dashboard.png'
os.makedirs(FIGURES_DIR, exist_ok=True)
# exist_ok=True means: don't raise an error if the directory already exists

fig.savefig(CHART_PATH, dpi=150, bbox_inches='tight')
# dpi=150       — 150 dots per inch — high enough for a crisp PDF or slide
# bbox_inches='tight' — crop the saved image to the actual plot content

plt.show()
print(f'\n✓ Chart saved → {CHART_PATH}')


In [367]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 24: Save the cleaned CSV + write the JSON quality report
# ═══════════════════════════════════════════════════════════════════════════

import os

# ─── PART A: Save cleaned CSV ───────────────────────────────────────────────────────
EXPORT_COLS = [
    # ── Unique identifier ──────────────────────────────────────────────────
    '_id',

    # ── Personal information ────────────────────────────────────────────
    'full_name',
    'email',
    'ssn',
    'ip_address',
    'gender',          # CLEANED: standardised to Male/Female/NaN
    'gender_raw',      # ORIGINAL: M/Male/F/Female/''/None
    'date_of_birth',   # CLEANED: parsed to ISO 8601
    'date_of_birth_raw',
    'age',             # DERIVED: from date_of_birth vs AUDIT_DATE
    'zip_code',

    # ── Financial information ───────────────────────────────────────────
    'annual_income',        # CLEANED: numeric, imputed if null
    'annual_income_raw',    # ORIGINAL
    'income_imputed',       # FLAG: True = median was substituted
    'credit_history_months',      # CLEANED: abs() applied
    'credit_history_months_flag', # FLAG: True = was negative
    'debt_to_income',      # CLEANED: divided by 10 for 1 record
    'dti_flag',            # FLAG: True = decimal error corrected
    'savings_balance',

    # ── Loan decision ─────────────────────────────────────────────────
    'loan_approved',
    'interest_rate',
    'approved_amount',
    'rejection_reason',

    # ── Spending ───────────────────────────────────────────────────────
    'total_spending',
    'alcohol_spend',

    # ── Audit flags ───────────────────────────────────────────────────
    'email_missing',    # True = email was blank or null
    'ssn_missing',      # True = SSN was null
    'dob_missing',      # True = date_of_birth was null
    # ssn_conflict: True = this SSN is shared with a DIFFERENT application
    #   Case B (same name)  — same person applied twice
    #   Case C (diff name)  — identity conflict / possible fraud
    # Case A (same _id) is excluded — it was just an exact duplicate row
    'ssn_conflict',
]

# Create the export DataFrame and format the date column as a plain string
df_export = df[EXPORT_COLS].copy()
df_export['date_of_birth'] = df_export['date_of_birth'].dt.strftime('%Y-%m-%d')

df_export.to_csv(CLEAN_PATH, index=False)
print(f'Export shape  : {df_export.shape[0]} rows × {df_export.shape[1]} columns')
print(f'✓ Cleaned CSV saved → {CLEAN_PATH}')
print()
print('Flag columns in the output:')
for col in [c for c in EXPORT_COLS if c.endswith(('_flag', '_missing', '_imputed', '_conflict'))]:
    n = int(df_export[col].sum())
    print(f'  {col:<35} True in {n:>3} records')

print()

# ─── PART B: Write the JSON quality report ────────────────────────────────────────═
# The report is the formal governance evidence artefact read by the
# Data Governance Officer. It records WHAT was wrong, HOW MANY records
# were affected, WHAT was done, and WHEN the audit ran.

n_ssn_conflict = int(df['ssn_conflict'].sum())

report_issues = [
    {'issue': '#1 — Duplicate Records (_id)',       'count': len(df_raw) - len(df),         'dimension': 'Uniqueness',      'fix': 'keep=first dedup'},
    {'issue': '#2 — Inconsistent Gender Coding',    'count': int(df['gender_raw'].isin(['M','F']).sum()), 'dimension': 'Consistency',     'fix': 'M→Male, F→Female via lookup dict'},
    {'issue': '#3 — Income as Text',               'count': int(string_income_mask.sum()),  'dimension': 'Validity / Type', 'fix': 'pd.to_numeric(errors=coerce)'},
    {'issue': '#4 — Missing Income',               'count': int(df['income_imputed'].sum()),'dimension': 'Completeness',    'fix': 'Median imputation + income_imputed flag'},
    {'issue': '#5 — Negative Credit History',      'count': int(df['credit_history_months_flag'].sum()), 'dimension': 'Validity', 'fix': 'abs() applied'},
    {'issue': '#6 — DTI Ratio > 1.0',              'count': int(df['dti_flag'].sum()),      'dimension': 'Validity',        'fix': 'Divided by 10'},
    {'issue': '#7 — Mixed Date Formats',           'count': int((df['dob_format'] != 'YYYY-MM-DD').sum()), 'dimension': 'Consistency', 'fix': 'Multi-format parser → ISO 8601'},
    {'issue': '#8 — Missing Date of Birth',        'count': int(df['dob_missing'].sum()),   'dimension': 'Completeness',    'fix': 'Flag only — no PII fabrication'},
    {'issue': '#9 — Missing Email',               'count': int(df['email_missing'].sum()),  'dimension': 'Completeness',    'fix': 'Flag only — no PII fabrication'},
    {'issue': '#10 — Missing SSN',                'count': int(df['ssn_missing'].sum()),    'dimension': 'Completeness',    'fix': 'Flag only — no PII fabrication'},
    {'issue': '#11 — Duplicate / Conflicting SSN', 'count': n_ssn_conflict,                 'dimension': 'Uniqueness',      'fix': 'Classified A/B/C + ssn_conflict flag; Cases B/C escalated'},
]

report = {
    'meta': {
        'project'      : 'NovaCred Credit Application Governance Audit',
        'notebook'     : 'Data-Quality-01.ipynb',
        'audit_date'   : AUDIT_DATE.strftime('%Y-%m-%d'),
        'generated_at' : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'raw_file'     : 'raw_credit_applications.json',
        'clean_file'   : 'cleaned_credit_applications.csv',
    },
    'summary': {
        'raw_records'         : len(df_raw),
        'duplicate_id_removed': len(df_raw) - len(df),
        'clean_records'       : len(df),
        'issues_catalogued'   : len(report_issues),
        'output_columns'      : df_export.shape[1],
    },
    'issues': report_issues,
}

os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
with open(REPORT_PATH, 'w', encoding='utf-8') as fh:
    json.dump(report, fh, indent=4, ensure_ascii=False)


Export shape  : 500 rows × 29 columns
✓ Cleaned CSV saved → /Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/data/cleaned_credit_applications.csv

Flag columns in the output:
  income_imputed                      True in   5 records
  credit_history_months_flag          True in   2 records
  dti_flag                            True in   1 records
  email_missing                       True in   7 records
  ssn_missing                         True in   4 records
  dob_missing                         True in   4 records
  ssn_conflict                        True in   4 records



---
## Section 14 — Complete Pipeline Diagram

The entire data engineering workflow in one picture:

```
┌─────────────────────────────────────────────────────────────────┐
│              RAW FILE: raw_credit_applications.json             │
│                   502 nested JSON records                       │
└───────────────────────────┬─────────────────────────────────────┘
                            │
                            ▼
             ┌──────────────────────────┐
             │  STEP 1: LOAD & FLATTEN  │
             │  • Open JSON file        │
             │  • Extract all nested    │
             │    fields into columns   │
             │  • Summarise spending    │
             │    array into totals     │
             └──────────────┬───────────┘
                            │
                            ▼
           ┌────────────────────────────────┐
           │  STEP 2: FIX DATA QUALITY      │
           │                                │
           │  #1 Remove Dups and Flag       |
           │     (keep='first')             │
           │                                │
           │  #2 Standardise gender         │
           │     M/F → Male/Female          │
           │                                │
           │  #3 Fix string incomes         │
           │     "55000" → 55000            │
           │                                │
           │  #4 Impute missing income      │
           │     null → median + flag       │
           │                                │
           │  #5 Fix negative credit        │
           │     -10 → 10 (abs value)       │
           │                                │
           │  #6 Fix impossible DTI         │
           │     1.85 → 0.185 (÷ 10)        │
           │                                │
           │  #7 Unify date formats         │
           │     3 formats → ISO 8601       │
           │                                │
           │  #8 Flag missing DOB           │
           │     null → NaN + flag          │
           │                                │
           │  #9 Flag missing PII           │
           │     email/SSN → flag           │
           └────────────────┬───────────────┘
                            │
                            ▼
     ┌──────────────────────────────────────────────┐
     │              OUTPUT FILE                     │
     │                                              │
     │     cleaned_credit_applications.csv          │
     │     500 rows × 28 columns                    │
     │     • Clean columns for analysis             │
     │     • Raw columns as audit trail             │
     │     • Flag columns for transparency          │
     │     ↓                                        │
     │     INPUT for Notebook 02 (Bias Analysis)    │
     │     INPUT for Notebook 03 (Privacy Audit)    │
     │                                              │
     └──────────────────────────────────────────────┘
```

---

